# Monte Carlo Within Metropolis Algorithm

In `metropolis_hastings.ipynb`, we saw how to sample from a target distribution by evaluating its density up to a normalizing constant. In `metropolis-within-gibbs.ipynb`, we combined exact conditional draws with Metropolis Hastings updates. We now consider what happens when even the likelihood evaluations needed for a Metropolis Hastings update are too difficult to calculate directly.


## Motivation

The ordinary Metropolis Hastings (MH) algorithm is useful when sampling from the target distribution is hard but evaluating its density up to proportionality is possible. For the posterior distribution,

$$
\pi(\theta\mid y) = \frac{p(y\mid\theta)p(\theta)}{p(y)},
$$

the normalizing constant $p(y)$ can be intractable without preventing us from using MH. Since it does not depend on $\theta$, it cancels from the posterior density ratio:

$$
\frac{\pi(\theta^*\mid y)}{\pi(\theta\mid y)}
= \frac{p(y\mid\theta^*)p(\theta^*)}{p(y\mid\theta)p(\theta)}.
$$

We still require evaluations of the likelihood and prior. The additional difficulty appears when the likelihood $p(y\mid\theta)$ is itself intractable. This can happen when a model contains an unobserved variable $z$ that must be integrated out:

$$
p(y\mid\theta) = \int p(y,z\mid\theta)\,dz.
$$

Unlike $p(y)$, this integral depends on $\theta$ and does not generally cancel. Monte Carlo within Metropolis (MCWM) addresses this difficulty by replacing the likelihood evaluations with Monte Carlo estimates. It produces an approximation to the desired MH algorithm and generally sacrifices exactness of the stationary target. We will assume throughout that the prior can be evaluated.

## Monte Carlo Likelihood Estimation

We first hold $\theta$ fixed and write the likelihood as an expectation:

$$
L(\theta) := p(y\mid\theta)
= \int p(y\mid z,\theta)p(z\mid\theta)\,dz
= \mathbb{E}_{Z\mid\theta}\left[p(y\mid Z,\theta)\right].
$$

If we can draw $Z^{(1)},\ldots,Z^{(N)}$ independently from $p(z\mid\theta)$, then we can estimate this expectation using

$$
\widehat L_N(\theta)
= \frac{1}{N}\sum_{k=1}^{N}p(y\mid Z^{(k)},\theta).
$$

Each draw gives us a possible hidden state. We evaluate the likelihood of the observed data given that state, then average these evaluations. The randomness comes from the simulated hidden states, while $y$ and $\theta$ remain fixed.

As in our importance sampling work, we can instead draw from a density $g_\theta(z)$. We require that this density can be sampled and evaluated, and is positive wherever $p(y,z\mid\theta)>0$. Then

$$
L(\theta)
= \mathbb{E}_{Z\sim g_\theta}\left[\frac{p(y,Z\mid\theta)}{g_\theta(Z)}\right],
$$

so, for independent draws $Z^{(k)}\sim g_\theta$, our estimator becomes

$$
\widehat L_N(\theta)
= \frac{1}{N}\sum_{k=1}^{N}w_\theta(Z^{(k)}),
\qquad
w_\theta(z) = \frac{p(y,z\mid\theta)}{g_\theta(z)}.
$$

Notice that this is an ordinary average of importance weights, rather than a self-normalized importance sampling estimate. Choosing $g_\theta(z)=p(z\mid\theta)$ recovers the first estimator. The importance density $g_\theta$ is separate from the parameter proposal distribution used by MH.

Provided the likelihood integral is finite, the estimator is unbiased and consistent:

$$
\mathbb{E}\left[\widehat L_N(\theta)\right]=L(\theta),
\qquad
\widehat L_N(\theta)\xrightarrow{\mathrm{a.s.}}L(\theta)
\quad\text{as }N\to\infty.
$$

If the importance weights have finite variance, then

$$
\operatorname{Var}\left[\widehat L_N(\theta)\right]
= \frac{\operatorname{Var}_{g_\theta}\left[w_\theta(Z)\right]}{N}.
$$

We will assume strictly positive likelihood estimates on the posterior support so that their ratios and logarithms are defined. Although we will work on the log scale for numerical stability, taking the logarithm does not preserve unbiasedness. In general,

$$
\mathbb{E}\left[\log\widehat L_N(\theta)\right]
\ne \log L(\theta).
$$

## MCWM Update

Starting at the current state $\theta^{t-1}$, one MCWM iteration proceeds as follows:

1. Propose $\theta^*\sim q(\cdot\mid\theta^{t-1})$.
2. Generate a fresh likelihood estimate at $\theta^{t-1}$ and another at $\theta^*$, using $N$ simulated hidden states for each. Conditional on the two parameter values, these simulations are independent of each other and of previously generated likelihood estimates.
3. Calculate the noisy acceptance ratio and accept or reject the proposal.

Writing these fresh estimates as $\widehat L_{N,t}^{\mathrm{curr}}$ and $\widehat L_{N,t}^{\mathrm{prop}}$, the ratio is

$$
\widehat r_{N,t}
= \frac{\widehat L_{N,t}^{\mathrm{prop}}\,p(\theta^*)\,q(\theta^{t-1}\mid\theta^*)}
{\widehat L_{N,t}^{\mathrm{curr}}\,p(\theta^{t-1})\,q(\theta^*\mid\theta^{t-1})},
$$

with acceptance probability

$$
\widehat\alpha_{N,t}=\min(1,\widehat r_{N,t}).
$$

We draw $U_t\sim\operatorname{Uniform}(0,1)$ independently and set $\theta^t=\theta^*$ if $U_t<\widehat\alpha_{N,t}$. Otherwise, we retain $\theta^t=\theta^{t-1}$.

As before, we can make the decision on the log scale:

$$
\begin{aligned}
\log\widehat r_{N,t}
&= \log\widehat L_{N,t}^{\mathrm{prop}}-\log\widehat L_{N,t}^{\mathrm{curr}}\\
&\quad+\log p(\theta^*)-\log p(\theta^{t-1})\\
&\quad+\log q(\theta^{t-1}\mid\theta^*)-\log q(\theta^*\mid\theta^{t-1}).
\end{aligned}
$$

We accept when $\log U_t<\min(0,\log\widehat r_{N,t})$. For a symmetric proposal, the proposal density terms cancel.

The defining feature of MCWM is that both likelihood estimates are refreshed at every iteration, even after a rejection. We retain the parameter value when a proposal is rejected, but do not retain its likelihood estimate for the next iteration. This differs from the pseudo-marginal algorithm, which retains the current estimate after rejection. See Table 1 and the discussion following it in [Andrieu and Roberts (2009)](https://arxiv.org/pdf/0903.5480).

## The Resulting Markov Chain

For a fixed $N$, proposal rule, and likelihood-estimation procedure, the next state depends on the past only through the current parameter value. All additional random quantities are generated afresh. Therefore, the parameter sequence $\{\theta^t\}$ is a time-homogeneous Markov chain.

For fixed current and proposed values $\theta$ and $\theta^*$, let $\widehat r_N(\theta,\theta^*)$ denote the random ratio above. Averaging over the fresh likelihood estimates gives the probability of accepting this proposal:

$$
\overline\alpha_N(\theta,\theta^*)
= \mathbb{E}\left[\min\left(1,\widehat r_N(\theta,\theta^*)\right)\right].
$$

This averaged acceptance probability determines the transitions of the parameter chain. If the chain is ergodic, let $\pi_N$ denote its invariant distribution. This is the distribution approached by a sufficiently long MCWM run; it is generally different from the intended posterior $\pi$.

## Approximation Error

Unbiased likelihood estimates do not make the MCWM acceptance probability unbiased. Taking a ratio and applying $\min(1,\cdot)$ are nonlinear operations, so in general

$$
\mathbb{E}\left[\min\left(1,\widehat r_N(\theta,\theta^*)\right)\right]
\ne \min\left(1,r(\theta,\theta^*)\right),
$$

where $r$ is the acceptance ratio calculated using the exact likelihood. Consequently, the usual MH detailed-balance argument for $\pi$ does not generally apply to MCWM.

We should distinguish two sources of error. A finite chain has Monte Carlo error when estimating expectations under $\pi_N$. Separately, there is approximation error because $\pi_N$ may differ from $\pi$. If the chain is ergodic, running it for more iterations reduces the first error but does not remove the second at a fixed $N$.

Refreshing the current estimate can reduce sticking caused by retaining an unusually large likelihood estimate. However, better mixing and even stability are not automatic. Conditions for stability and convergence of $\pi_N$ toward $\pi$ are studied in [Medina-Aguayo, Lee and Roberts (2016)](https://link.springer.com/article/10.1007/s11222-015-9604-3), particularly Sections 1.3 and 4.

## Monte Carlo Sample Size

The sample size $N$ controls how much simulation is used for each likelihood estimate. It is separate from the number of iterations of the parameter chain. With independent, finite-variance importance weights, increasing $N$ reduces the likelihood-estimation variance at rate $1/N$, but also increases the work per MCWM iteration.

As the estimates become more accurate, the noisy acceptance probability approaches that of exact-likelihood MH for fixed parameter values with positive likelihoods. Convergence of the invariant distributions $\pi_N$ to $\pi$ requires additional assumptions ensuring that the chains remain well behaved as $N$ increases; pointwise consistency of the likelihood estimator alone is not sufficient.

For our first simulation, we will compare MCWM with exact-likelihood MH on a model where the likelihood is known. We can then vary $N$ and examine the posterior estimates, acceptance rates, autocorrelation, and computation time. The objective is to balance the accuracy of the posterior approximation with the cost of generating the chain.

In [1]:
# CODE